# Khmer Handwriting Recognition — Export for the Backend

In [1]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

CLASSICAL_MODELS_DIR = Path("../../models/classical")
QUANTUM_MODELS_DIR = Path("../../models/quantum")
TABLES_DIR = Path("../../results/tables/training")
BACKEND_MODELS_DIR = Path("../../website/backend/models")
BACKEND_MODELS_DIR.mkdir(parents=True, exist_ok=True)

QUBIT_SIZES = [4, 6, 8]


def read_csv_or_none(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print(f"{path.name} not found -- manifest will skip those accuracy numbers")
        return None


summary_df = read_csv_or_none(TABLES_DIR / "training_summary_alphabet.csv")
cnn_summary_df = read_csv_or_none(TABLES_DIR / "cnn_alphabet_training_summary.csv")

## Verify required files exist, then copy into the bundle

In [2]:
manifest = {"exported_at": datetime.now(timezone.utc).isoformat(), "qubit_counts": {}}

BACKEND_CLASSICAL_DIR = BACKEND_MODELS_DIR / "classical"
BACKEND_QUANTUM_DIR = BACKEND_MODELS_DIR / "quantum"

for n_qubits in QUBIT_SIZES:
    name = f"{n_qubits}q"
    cnn_path = CLASSICAL_MODELS_DIR / name / "cnn.pt"
    label_path = CLASSICAL_MODELS_DIR / name / "label_map.json"
    quantum_path = QUANTUM_MODELS_DIR / name / "best.npz"

    missing = [p for p in [cnn_path, label_path, quantum_path] if not p.exists()]
    if missing:
        print(f"{n_qubits}q: SKIPPED -- missing {[str(p) for p in missing]}")
        continue

    (BACKEND_CLASSICAL_DIR / name).mkdir(parents=True, exist_ok=True)
    (BACKEND_QUANTUM_DIR / name).mkdir(parents=True, exist_ok=True)
    shutil.copy(cnn_path, BACKEND_CLASSICAL_DIR / name / "cnn.pt")
    shutil.copy(label_path, BACKEND_CLASSICAL_DIR / name / "label_map.json")
    shutil.copy(quantum_path, BACKEND_QUANTUM_DIR / name / "best.npz")

    with open(label_path, encoding="utf-8") as f:
        n_classes = len(json.load(f))

    entry = {"n_classes": n_classes}
    if summary_df is not None:
        row = summary_df[summary_df["qubits"] == n_qubits]
        # 02_train_hybrid.ipynb now sweeps several layer counts per qubit count -- the deployed
        # checkpoint (best.npz, the one we just copied) is always the one trained with the
        # largest swept layer count, so pick that row's numbers.
        if "layers" in row.columns and len(row):
            row = row[row["layers"] == row["layers"].max()]
        if len(row):
            entry["val_accuracy"] = float(row["best_val_accuracy"].iloc[0])
            entry["test_accuracy"] = float(row["test_accuracy"].iloc[0])

    manifest["qubit_counts"][str(n_qubits)] = entry
    print(f"{n_qubits}q: exported ({n_classes} classes)")

baseline_cnn = CLASSICAL_MODELS_DIR / "baseline" / "cnn.pt"
baseline_labels = CLASSICAL_MODELS_DIR / "baseline" / "label_map.json"
missing = [str(p) for p in [baseline_cnn, baseline_labels] if not p.exists()]
if missing:
    print(f"classical baseline: SKIPPED -- missing {missing} (website falls back to each qubit count's own CNN head)")
else:
    (BACKEND_CLASSICAL_DIR / "baseline").mkdir(parents=True, exist_ok=True)
    shutil.copy(baseline_cnn, BACKEND_CLASSICAL_DIR / "baseline" / "cnn.pt")
    shutil.copy(baseline_labels, BACKEND_CLASSICAL_DIR / "baseline" / "label_map.json")
    with open(baseline_labels, encoding="utf-8") as f:
        entry = {"n_classes": len(json.load(f))}
    if cnn_summary_df is not None and "model" in cnn_summary_df.columns:
        row = cnn_summary_df[cnn_summary_df["model"] == "baseline"]
        if len(row):
            entry["val_accuracy"] = float(row["best_val_accuracy"].iloc[0])
            entry["test_accuracy"] = float(row["test_accuracy"].iloc[0])
    manifest["classical_baseline"] = entry
    print(f"classical baseline: exported ({entry['n_classes']} classes)")

with open(BACKEND_MODELS_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Wrote:", BACKEND_MODELS_DIR / "manifest.json")
manifest

4q: exported (16 classes)
6q: exported (33 classes)
8q: exported (33 classes)
classical baseline: exported (33 classes)
Wrote: ..\..\website\backend\models\manifest.json


{'exported_at': '2026-11-24T05:53:43.854119+00:00',
 'qubit_counts': {'4': {'n_classes': 16,
   'val_accuracy': 0.9323,
   'test_accuracy': 0.8854},
  '6': {'n_classes': 33, 'val_accuracy': 0.9091, 'test_accuracy': 0.899},
  '8': {'n_classes': 33, 'val_accuracy': 0.9091, 'test_accuracy': 0.9242}},
 'classical_baseline': {'n_classes': 33,
  'val_accuracy': 0.9495,
  'test_accuracy': 0.9318}}

## Sanity check — list what's now in the bundle

In [3]:
for p in sorted(BACKEND_MODELS_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(BACKEND_MODELS_DIR))

  classical\4q\cnn.pt
  classical\4q\label_map.json
  classical\6q\cnn.pt
  classical\6q\label_map.json
  classical\8q\cnn.pt
  classical\8q\label_map.json
  classical\baseline\cnn.pt
  classical\baseline\label_map.json
  manifest.json
  quantum\4q\best.npz
  quantum\6q\best.npz
  quantum\8q\best.npz
